In [15]:
import torch

class KroneckerProduct:
    def __init__(self):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    def compute(self, A: torch.Tensor, B: torch.Tensor) -> torch.Tensor:
        """
        Compute Kronecker product A ⊗ B.
        For matrices A (m×n) and B (p×q), result is (mp×nq).
        """
        # Move tensors to GPU
        A = A.to(self.device)
        B = B.to(self.device)

        # Get dimensions
        m, n = A.shape
        p, q = B.shape

        # Initialize output
        result = torch.zeros((m * p, n * q), device=self.device)

        # Compute Kronecker product
        for i in range(m):
            for j in range(n):
                result[i*p:(i+1)*p, j*q:(j+1)*q] = A[i, j] * B

        return result

def benchmark_comparison(A: torch.Tensor, B: torch.Tensor, num_runs: int = 3):
    """
    Compare performance and results between custom and torch.kron implementations.
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    A = A.to(device)
    B = B.to(device)

    # Custom implementation
    kron = KroneckerProduct()
    custom_times = []

    print("\nCustom Implementation:")
    for i in range(num_runs):
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)

        start.record()
        custom_result = kron.compute(A, B)
        end.record()

        torch.cuda.synchronize()
        custom_times.append(start.elapsed_time(end))

        if i == 0:  # Save first result for comparison
            first_custom_result = custom_result

    # PyTorch implementation
    torch_times = []

    print("\nPyTorch Implementation:")
    for i in range(num_runs):
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)

        start.record()
        torch_result = torch.kron(A, B)
        end.record()

        torch.cuda.synchronize()
        torch_times.append(start.elapsed_time(end))

        if i == 0:  # Save first result for comparison
            first_torch_result = torch_result

    # Compare results
    max_diff = torch.max(torch.abs(first_custom_result - first_torch_result)).item()
    is_close = torch.allclose(first_custom_result, first_torch_result, rtol=1e-5, atol=1e-8)

    # Report results
    results = {
        'custom_time_avg': sum(custom_times) / len(custom_times),
        'custom_time_min': min(custom_times),
        'torch_time_avg': sum(torch_times) / len(torch_times),
        'torch_time_min': min(torch_times),
        'max_difference': max_diff,
        'results_match': is_close
    }

    return results


# Test with different sizes
if __name__ == "__main__":
    test_sizes = [
        ((5, 5), (5, 5)),      # Tiny
        ((10, 10), (5, 5)),    # Small
        ((50, 50), (10, 10)),  # Medium
        ((100, 100), (50, 50)),  # Larger
    ]

    for (m, n), (p, q) in test_sizes:
        print(f"\n{'='*60}")
        print(f"Testing with A: {m}×{n}, B: {p}×{q}")
        print(f"Output size will be: {m*p}×{n*q}")

        try:
            # Create random matrices
            A = torch.randn(m, n)
            B = torch.randn(p, q)

            # Run benchmark
            results = benchmark_comparison(A, B)

            # Print results
            print(f"\nResults:")
            print(f"Custom Implementation:")
            print(f"  Average time: {results['custom_time_avg']:.2f} ms")
            print(f"  Best time: {results['custom_time_min']:.2f} ms")
            print(f"\nPyTorch Implementation:")
            print(f"  Average time: {results['torch_time_avg']:.2f} ms")
            print(f"  Best time: {results['torch_time_min']:.2f} ms")
            print(f"\nAccuracy:")
            print(f"  Results match: {results['results_match']}")
            print(f"  Maximum difference: {results['max_difference']:.2e}")

            # Memory usage
            if torch.cuda.is_available():
                memory_gb = torch.cuda.max_memory_allocated() / (1024**3)
                print(f"\nPeak GPU memory usage: {memory_gb:.2f} GB")

            # Clear memory
            torch.cuda.empty_cache()

        except RuntimeError as e:
            if "out of memory" in str(e):
                print(f"Out of memory error - matrix too large")
            else:
                print(f"Error: {str(e)}")


Testing with A: 5×5, B: 5×5
Output size will be: 25×25

Custom Implementation:

PyTorch Implementation:

Results:
Custom Implementation:
  Average time: 4.68 ms
  Best time: 2.11 ms

PyTorch Implementation:
  Average time: 0.05 ms
  Best time: 0.01 ms

Accuracy:
  Results match: True
  Maximum difference: 0.00e+00

Peak GPU memory usage: 0.14 GB

Testing with A: 10×10, B: 5×5
Output size will be: 50×50

Custom Implementation:

PyTorch Implementation:

Results:
Custom Implementation:
  Average time: 24.74 ms
  Best time: 22.20 ms

PyTorch Implementation:
  Average time: 0.11 ms
  Best time: 0.11 ms

Accuracy:
  Results match: True
  Maximum difference: 0.00e+00

Peak GPU memory usage: 0.14 GB

Testing with A: 50×50, B: 10×10
Output size will be: 500×500

Custom Implementation:

PyTorch Implementation:

Results:
Custom Implementation:
  Average time: 171.43 ms
  Best time: 137.52 ms

PyTorch Implementation:
  Average time: 0.44 ms
  Best time: 0.01 ms

Accuracy:
  Results match: True
  

In [17]:
# Example usage
A = torch.randn(50, 50, device='cuda')  # Put tensors directly on GPU
B = torch.randn(10, 10, device='cuda')

# Compute Kronecker product
result = torch.kron(A, B)

In [24]:
import torch
from torch.utils.cpp_extension import load_inline
import os
import tempfile
from pathlib import Path

# Create a temporary build directory
build_dir = Path(tempfile.gettempdir()) / 'cuda_kronecker_build'
build_dir.mkdir(exist_ok=True)
print(f"Using build directory: {build_dir}")

# Set CUDA architecture
os.environ['TORCH_CUDA_ARCH_LIST'] = '8.6'  # Adjust this for your GPU

# CUDA kernel implementation
cuda_source = """
extern "C" __global__ void kronecker_product_kernel(
    const float* __restrict__ A,
    const float* __restrict__ B,
    float* __restrict__ output,
    const int batch_size,
    const int Ar, const int Ac,
    const int Br, const int Bc,
    const int Or, const int Oc) {

    const int tid = blockIdx.x * blockDim.x + threadIdx.x;
    const int batch_idx = blockIdx.y;

    const int total_elements = Or * Oc;

    if (tid >= total_elements) return;

    // Calculate output position
    const int o_row = tid / Oc;
    const int o_col = tid % Oc;

    // Calculate positions in A and B
    const int a_row = o_row / Br;
    const int a_col = o_col / Bc;
    const int b_row = o_row % Br;
    const int b_col = o_col % Bc;

    // Calculate offsets
    const int a_offset = batch_idx * Ar * Ac + a_row * Ac + a_col;
    const int b_offset = batch_idx * Br * Bc + b_row * Bc + b_col;

    // Write output
    output[batch_idx * total_elements + tid] = A[a_offset] * B[b_offset];
}
"""

cpp_source = """
#include <torch/extension.h>
#include <cuda_runtime.h>

torch::Tensor kronecker_product_cuda(
    const torch::Tensor& A,
    const torch::Tensor& B) {

    const int batch_size = A.size(0);
    const int Ar = A.size(1), Ac = A.size(2);
    const int Br = B.size(1), Bc = B.size(2);
    const int Or = Ar * Br, Oc = Ac * Bc;

    auto output = torch::zeros({batch_size, Or, Oc},
                             torch::TensorOptions()
                                 .dtype(torch::kFloat32)
                                 .device(A.device()));

    const int threads = 256;
    const int blocks = (Or * Oc + threads - 1) / threads;
    const dim3 grid(blocks, batch_size);
    const dim3 block(threads);

    kronecker_product_kernel<<<grid, block>>>(
        A.data_ptr<float>(),
        B.data_ptr<float>(),
        output.data_ptr<float>(),
        batch_size, Ar, Ac, Br, Bc, Or, Oc);

    return output;
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("kronecker_product_cuda", &kronecker_product_cuda,
          "Kronecker product CUDA implementation");
}
"""

def load_cuda_extension():
    """Load CUDA extension with error handling"""
    try:
        cuda_module = load_inline(
            name='kronecker_product_cuda',
            cpp_sources=cpp_source,
            cuda_sources=cuda_source,
            functions=['kronecker_product_cuda'],
            with_cuda=True,
            extra_cuda_cflags=['-O3'],
            build_directory=str(build_dir),
            verbose=True
        )
        return cuda_module
    except Exception as e:
        print(f"Error loading CUDA extension:")
        print(f"Build directory: {build_dir}")
        print(f"Error message: {str(e)}")
        raise

class CUDAKroneckerProduct(torch.nn.Module):
    def __init__(self):
        super(CUDAKroneckerProduct, self).__init__()
        self.cuda_module = load_cuda_extension()

    def forward(self, A: torch.Tensor, B: torch.Tensor) -> torch.Tensor:
        """
        Compute Kronecker product using CUDA kernel.

        Args:
            A: Input tensor of shape (batch_size, m, n)
            B: Input tensor of shape (batch_size, p, q)

        Returns:
            Output tensor of shape (batch_size, m*p, n*q)
        """
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA is not available")

        assert A.is_cuda and B.is_cuda, "Inputs must be CUDA tensors"
        assert A.dtype == torch.float32 and B.dtype == torch.float32, "Inputs must be float32"
        return self.cuda_module.kronecker_product_cuda(A, B)

def benchmark(kron_cuda, A: torch.Tensor, B: torch.Tensor, num_warmup: int = 10, num_iters: int = 100):
    """Benchmark the CUDA implementation"""
    device = A.device

    # Warmup
    with torch.no_grad():
        for _ in range(num_warmup):
            _ = kron_cuda(A, B)
    torch.cuda.synchronize()

    # Timing
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    with torch.no_grad():
        start_event.record()
        for _ in range(num_iters):
            result = kron_cuda(A, B)
        end_event.record()

    torch.cuda.synchronize()
    elapsed_time = start_event.elapsed_time(end_event) / num_iters

    return elapsed_time, result

if __name__ == "__main__":
    # Test configuration
    batch_size = 32
    A_shape = (100, 100)
    B_shape = (20, 20)

    try:
        print("\nInitializing CUDA Kronecker Product module...")
        kron_cuda = CUDAKroneckerProduct().cuda()

        print("\nCreating test inputs...")
        A = torch.randn(batch_size, *A_shape, dtype=torch.float32, device='cuda')
        B = torch.randn(batch_size, *B_shape, dtype=torch.float32, device='cuda')

        print(f"\nRunning benchmark:")
        print(f"Batch size: {batch_size}")
        print(f"A shape: {A_shape}")
        print(f"B shape: {B_shape}")

        elapsed_time, result = benchmark(kron_cuda, A, B)

        print(f"\nResults:")
        print(f"Average time per iteration: {elapsed_time:.3f} ms")
        print(f"Output shape: {result.shape}")
        print(f"Memory usage: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

    except Exception as e:
        print(f"\nError during execution:")
        print(str(e))
        import traceback
        traceback.print_exc()

    finally:
        print("\nCleaning up...")
        torch.cuda.empty_cache()

        # Optional: Clean up build directory
        # import shutil
        # shutil.rmtree(build_dir, ignore_errors=True)

Using build directory: C:\Users\hello\AppData\Local\Temp\cuda_kronecker_build

Initializing CUDA Kronecker Product module...
Error loading CUDA extension:
Build directory: C:\Users\hello\AppData\Local\Temp\cuda_kronecker_build
Error message: Command '['where', 'cl']' returned non-zero exit status 1.

Error during execution:
Command '['where', 'cl']' returned non-zero exit status 1.

Cleaning up...


The input conditions for extension module kronecker_product_cuda have changed. Bumping to version 1 and re-building as kronecker_product_cuda_v1...
Detected CUDA files, patching ldflags
Emitting ninja build file C:\Users\hello\AppData\Local\Temp\cuda_kronecker_build\build.ninja...
Traceback (most recent call last):
  File "C:\Users\hello\AppData\Local\Temp\ipykernel_27956\1807550197.py", line 166, in <module>
    kron_cuda = CUDAKroneckerProduct().cuda()
                ^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\hello\AppData\Local\Temp\ipykernel_27956\1807550197.py", line 113, in __init__
    self.cuda_module = load_cuda_extension()
                       ^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\hello\AppData\Local\Temp\ipykernel_27956\1807550197.py", line 93, in load_cuda_extension
    cuda_module = load_inline(
                  ^^^^^^^^^^^^
  File "C:\Users\hello\anaconda3\envs\research\Lib\site-packages\torch\utils\cpp_extension.py", line 1646, in load_inline
    return _jit_compile(
 